`Fuzzy_neural_network_CP.ipynb`
`Neuro‑Fuzzy (TSK/ANFIS‑style)` Regressor to predict Collapse Potential (%)
from soil experiment features.

What we have in this single script
----------------------------------
• Clean data loading.

• Feature scaling (StandardScaler on X only, fit on train).

• A first‑order Takagi–Sugeno–Kang (TSK) neuro‑fuzzy network implemented in PyTorch:

    - Gaussian membership functions (MFs) per input feature.
    - Grid partition to form rules (R = M^D, where M = MFs per feature, D = #features).
    - Normalized firing strengths and linear consequents per rule.

• Robust initialization of MF centers/sigmas from feature percentiles.

• AdamW optimizer + cosine schedule; early stopping on validation MSE.

• Full evaluation: RMSE, MAE, R², MAPE; residual analysis.

• Visualizations: training curves, parity plot, residual histogram, learning curves.

• Simple permutation feature importance on the validation set.

• Model checkpointing to ./artifacts/.


Notes
-----
• Default #MFs per feature is M=3 → with D=6 features gives R = 3^6 = 729 rules (tractable on CPU/GPU).

• For very modest machine, reduce M to 2.

• Tuning M, batch size, learning rate, epochs at the bottom of the script.


In [ ]:
import os
import math
import json
import time
import random
from dataclasses import dataclass
from typing import Tuple, List


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Config
DATA_CSV = "/content/drive/MyDrive/PINNs/Suction_vsCP-modified_1.xlsx"
TARGET_COL = "Collapse Potential (%)"
FEATURE_COLS = [
"Suction (kPa)",
"Silica fume (%)",
"Lime (%)",
"Gypsum content (%)",
"Applied vertical stress (kPa)",
"Degree of Saturation (%)",
]

ARTIFACTS_DIR = "/content/drive/MyDrive/NNsGA/FNNs/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Utilities

def rmse(y_true, y_pred):
  return math.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred, eps=1e-8):
  y_true = np.asarray(y_true)
  y_pred = np.asarray(y_pred)
  return np.mean(np.abs((y_true - y_pred) / (np.clip(np.abs(y_true), eps, None)))) * 100.0

def metrics(y_true, y_pred):
  return {
    "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    "MAE": float(mean_absolute_error(y_true, y_pred)),
    "R2": float(r2_score(y_true, y_pred)),
  }

In [ ]:
@dataclass
class TrainConfig:
    mfs_per_feature: int = 3 # M
    batch_size: int = 128
    max_epochs: int = 400
    lr: float       = 1e-3
    weight_decay: float = 1e-4
    patience: int   = 40    # early stopping
    warmup_epochs: int = 10

In [ ]:
# Data
class TabDataset(Dataset):
  def __init__(self, X: np.ndarray, y: np.ndarray):
    self.X = torch.from_numpy(X.astype(np.float32))
    self.y = torch.from_numpy(y.astype(np.float32)).view(-1, 1)

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [ ]:
# Neuro‑Fuzzy (TSK) Model
class TSKFuzzyRegressor(nn.Module):
    """First‑order TSK neuro‑fuzzy network with Gaussian MFs and grid rules.

    Input: x in R^D
    - For each feature j, we have M Gaussian MFs: mu_{j,m}(x_j) = exp(-0.5 * ((x_j - c_{j,m})/s_{j,m})^2)
    - Rules are the Cartesian product of feature MFs → R = M^D rules.
    - Firing strength w_r(x) = Π_j mu_{j, m_j}(x_j)
    - Consequent per rule r: y_r(x) = a_{r,0} + Σ_j a_{r,j} * x_j
    - Output: y(x) = Σ_r [ (w_r / Σ_k w_k) * y_r(x) ]
    """

    def __init__(self, D: int, M: int):
        super().__init__()
        self.D = D
        self.M = M
        self.R = M ** D

        # MF parameters per feature
        # centers: (D, M), sigmas: (D, M) (positivity via softplus)
        self.centers = nn.Parameter(torch.zeros(D, M))
        self.log_sigmas = nn.Parameter(torch.zeros(D, M))  # sigma = softplus(log_sigma)

        # Rule index tensor: (R, D) with values in [0, M-1]
        combos = np.stack(np.meshgrid(*[np.arange(M) for _ in range(D)], indexing='ij'), axis=-1).reshape(-1, D)
        self.register_buffer('rule_index', torch.from_numpy(combos).long())

        # Linear consequents per rule: a0 (bias) + a per feature
        self.consequents = nn.Linear(D, self.R, bias=True)  # will output (N, R) of Σ_j a_{r,j} x_j + a_{r,0}

        # small epsilon to stabilize normalization
        self.eps = 1e-8

    def gaussian_mf(self, x):
        """Compute membership values for all features & MFs.
        x: (N, D)
        return: mu of shape (N, D, M)
        """
        N, D = x.shape
        centers = self.centers  # (D, M)
        sigmas = torch.nn.functional.softplus(self.log_sigmas) + 1e-4  # (D, M)
        # expand for broadcasting
        x_exp = x.unsqueeze(-1)              # (N, D, 1)
        c_exp = centers.unsqueeze(0)        # (1, D, M)
        s_exp = sigmas.unsqueeze(0)         # (1, D, M)
        z = (x_exp - c_exp) / s_exp
        mu = torch.exp(-0.5 * z * z)        # (N, D, M)
        return mu

    def rule_firing(self, mu):
        """Compute rule firing strengths w_r via product across selected MFs.
        mu: (N, D, M)
        returns: w of shape (N, R)
        """
        N, D, M = mu.shape
        gather_list = []
        for j in range(D):
            mu_j = mu[:, j, :]                       # (N, M)
            mu_jg = mu_j.index_select(dim=1, index=self.rule_index[:, j]).view(N, -1)  # (N, R)
            gather_list.append(mu_jg)
        w = torch.ones_like(gather_list[0])
        for g in gather_list:
            w = w * g
        return w  # (N, R)

    def forward(self, x):
        # x: (N, D)
        mu = self.gaussian_mf(x)           # (N, D, M)
        w = self.rule_firing(mu)           # (N, R)
        w_sum = w.sum(dim=1, keepdim=True) # (N, 1)
        beta = w / (w_sum + self.eps)      # normalized firing strengths

        # linear consequents per rule for each sample
        # consequents(x): (N, R) representing Σ_j a_{r,j} x_j + a_{r,0}
        y_lin = self.consequents(x)        # (N, R)
        y = (beta * y_lin).sum(dim=1, keepdim=True)  # (N, 1)
        return y, w_sum

In [ ]:
# Initialization helpers

def init_mfs_from_data(model: TSKFuzzyRegressor, X_train: np.ndarray):
    """Initialize MF centers using feature percentiles and sigmas using spread."""
    D = X_train.shape[1]
    M = model.M
    for j in range(D):
        # centers from percentiles between 5th..95th
        perc = np.linspace(5, 95, M)
        c = np.percentile(X_train[:, j], perc)
        # ensure sorted and unique-ish
        c = np.unique(np.round(c, 6))
        if c.size < M:
            # pad by small jitter around median
            med = np.median(X_train[:, j])
            pad = np.linspace(-1, 1, M - c.size) * np.std(X_train[:, j]) * 0.1 + med
            c = np.sort(np.concatenate([c, pad]))
        s = np.full(M, np.std(X_train[:, j]) + 1e-3)
        with torch.no_grad():
            model.centers[j].copy_(torch.from_numpy(c.astype(np.float32)))
            model.log_sigmas[j].copy_(torch.log(torch.from_numpy(s.astype(np.float32))))

In [ ]:
# Training loop
def train_model(model, train_loader, val_loader, cfg: TrainConfig):
    model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, cfg.max_epochs - cfg.warmup_epochs))

    best_val = float('inf')
    best_state = None
    history = {"train": [], "val": [], "lr": []}
    patience = cfg.patience

    for epoch in range(1, cfg.max_epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            pred, _ = model(xb)
            loss = criterion(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_losses.append(loss.item())
        train_loss = float(np.mean(train_losses))

        # validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                pred, _ = model(xb)
                loss = criterion(pred, yb)
                val_losses.append(loss.item())
        val_loss = float(np.mean(val_losses))

        # LR scheduling (simple: step after warmup period)
        if epoch > cfg.warmup_epochs:
            scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        history["train"].append(train_loss)
        history["val"].append(val_loss)
        history["lr"].append(current_lr)

        # early stopping
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = cfg.patience
        else:
            patience -= 1
            if patience <= 0:
                break

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:04d} | train MSE={train_loss:.4f} | val MSE={val_loss:.4f} | lr={current_lr:.2e}")

    # restore best
    if best_state is not None:
        model.load_state_dict(best_state)

    return history

In [ ]:
# Evaluation helpers
def evaluate(model, X: np.ndarray, y: np.ndarray) -> Tuple[dict, np.ndarray]:
    model.eval()
    with torch.no_grad():
        X_t = torch.from_numpy(X.astype(np.float32)).to(DEVICE)
        y_hat, _ = model(X_t)
        y_hat = y_hat.cpu().numpy().reshape(-1)
    metrics = {
        "RMSE": rmse(y, y_hat),
        "MAE": mean_absolute_error(y, y_hat),
        "R2": r2_score(y, y_hat),
        "MAPE_%": float(mape(y, y_hat)), # Cast np.float32 to standard float
    }
    return metrics, y_hat

In [ ]:
def plot_training(history: dict, outdir: str):
    plt.figure()
    plt.plot(history["train"], label="Train MSE")
    plt.plot(history["val"], label="Val MSE")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.title("Training/Validation Loss")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, "loss_curves.png"), dpi=160)
    plt.close()

In [ ]:
def plot_parity(y_true: np.ndarray, y_pred: np.ndarray, outdir: str, split_name: str):
    plt.figure()
    plt.scatter(y_true, y_pred, s=14, alpha=0.7)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims)
    plt.xlabel("Actual Collapse Potential (%)")
    plt.ylabel("Predicted Collapse Potential (%)")
    plt.title(f"Parity Plot — {split_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"parity_{split_name.lower()}.png"), dpi=160)
    plt.close()

In [ ]:
def plot_residuals(y_true: np.ndarray, y_pred: np.ndarray, outdir: str, split_name: str):
    res = y_pred - y_true
    plt.figure()
    plt.hist(res, bins=40)
    plt.xlabel("Residual (Pred − True)")
    plt.ylabel("Count")
    plt.title(f"Residuals — {split_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"residuals_{split_name.lower()}.png"), dpi=160)
    plt.close()

In [ ]:
def permutation_feature_importance(model, X_val, y_val, scaler: StandardScaler, n_repeats: int = 8):
    # simple, model-agnostic permutation importance
    base_metrics, base_pred = evaluate(model, X_val, y_val)
    base_rmse = base_metrics["RMSE"]
    D = X_val.shape[1]
    importances = np.zeros(D)
    for j in range(D):
        worsens = []
        for _ in range(n_repeats):
            Xp = X_val.copy()
            np.random.shuffle(Xp[:, j])
            m, _ = evaluate(model, Xp, y_val)
            worsens.append(m["RMSE"] - base_rmse)
        importances[j] = np.mean(worsens)
    return importances

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

directory_path = '/content/drive/MyDrive/PINNs/'

if os.path.exists(directory_path):
    print(f"Contents of '{directory_path}':")
    for item in os.listdir(directory_path):
        print(item)
else:
    print(f"The directory '{directory_path}' does not exist. Please ensure Google Drive is mounted correctly and the path is valid.")

Contents of '/content/drive/MyDrive/PINNs/':
Suction_vsCP-modified_1.xlsx
ensemble_models.pth
CP_Predictions.xlsx
pinn_model_exp2.keras
exp2_scaler_X.save
exp2_scaler_y.save
PINNs_2_models
PINNs_Exp1_2
PINNs_Exp2_2
saved_models
9_7_25_best_ga_ann_model.pth
acl_9_7_25_best_ga_ann_model.pth
acl_10_7_25_best_ga_ann_model.pth
PINNs_Exp2_2_ICE
PINNs_Exp2_2_ICE_2_MC


In [ ]:
import os

# Unmount if previously mounted and force deletion of mount point directory
if os.path.exists('/content/drive'):
    try:
        from google.colab import drive
        drive.flush_and_unmount()
    except Exception as e:
        print(f"Could not unmount Google Drive: {e}")

    # Attempt to remove the directory if it still exists and is not empty
    if os.path.exists('/content/drive') and os.path.isdir('/content/drive'):
        print("Removing existing /content/drive directory...")
        try:
            os.system('rm -rf /content/drive') # Use rm -rf for forceful removal
        except Exception as e:
            print(f"Error removing /content/drive: {e}")

# Recreate the directory to ensure it's empty for mounting
if not os.path.exists('/content/drive'):
    os.makedirs('/content/drive')
    print("/content/drive directory recreated.")

# Now, try mounting again
from google.colab import drive
drive.mount('/content/drive')

/content/drive directory recreated.
Mounted at /content/drive


In [ ]:
import pandas as pd

# Re-loading the data to show the process clearly
data_path = "/content/drive/MyDrive/PINNs/Suction_vsCP-modified_1.xlsx"
df_loaded = pd.read_excel(data_path)
print("Successfully loaded data from:", data_path)
display(df_loaded.head())

Successfully loaded data from: /content/drive/MyDrive/PINNs/Suction_vsCP-modified_1.xlsx


,Suction (kPa),Silica fume (%),Lime (%),Gypsum content (%),Applied vertical stress (kPa),Degree of Saturation (%),Collapse Potential (%)
0,5,0,0,15,200.0,50.0,1.50
1,10,0,0,15,200.0,40.0,1.40
2,16,0,0,15,200.0,42.8,1.36
3,25,0,0,15,200.0,45.0,1.30
4,32,0,0,15,200.0,53.5,1.47


In [ ]:
# Revised main: 70/30 split, leakage-free preprocessing, 4-fold tuning
if __name__ == "__main__":
    # ============================================================
    # COLAB CONFIGURATION
    # ============================================================

    from sklearn.model_selection import KFold

    DATA_PATH = "/content/drive/MyDrive/PINNs/Suction_vsCP-modified_1.xlsx"

    OUT_DIR = "/content/drive/MyDrive/NNsGA/FNNs/GeoANN-GA_fuzzy_revised_results"

    EPOCHS = 250

    os.makedirs(OUT_DIR, exist_ok=True)

    df = pd.read_excel(DATA_PATH)

    # Removed redundant lines that were causing NameError and were leftovers from argparse setup
    # os.makedirs(args.out, exist_ok=True)
    # df = pd.read_excel(args.data)

    df = df[FEATURE_COLS + [TARGET_COL]].dropna().copy()
    X = df[FEATURE_COLS].values.astype(np.float32)
    y = df[TARGET_COL].values.astype(np.float32)

    # Raw split BEFORE fitting any transform.
    X_dev, X_test, y_dev, y_test = train_test_split(
        X, y, test_size=0.30, random_state=SEED
    )
    print(f"Development={len(X_dev)}, independent test={len(X_test)}")

    def preprocess_fit(Xtr):
        Xtr = Xtr.copy()
        Xtr[:, 0] = np.log1p(np.clip(Xtr[:, 0], 0, None))
        sc = StandardScaler()
        return sc, sc.fit_transform(Xtr)

    def preprocess_transform(Xv, sc):
        Xv = Xv.copy()
        Xv[:, 0] = np.log1p(np.clip(Xv[:, 0], 0, None))
        return sc.transform(Xv)

    def train_one(Xtr, ytr, Xv, yv, mfs, lr):
        sc, Xtrs = preprocess_fit(Xtr)
        Xvs = preprocess_transform(Xv, sc)
        model = TSKFuzzyRegressor(D=Xtrs.shape[1], M=mfs)
        # Initialize Gaussian membership functions from the training fold
        init_mfs_from_data(model, Xtrs)
        cfg = TrainConfig(
                          mfs_per_feature=mfs,
                          batch_size=64,
                          max_epochs=EPOCHS,
                          lr=lr,
                          weight_decay=1e-4,
                          patience=35,
                          warmup_epochs=10
                      )
        train_ds = TabDataset(Xtrs, ytr)
        val_ds = TabDataset(Xvs, yv)
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
        train_model(model, train_loader, val_loader, cfg)
        _, pred = evaluate(model, Xvs, yv) # Corrected line: assign predictions to pred
        return model, sc, pred

    # 4-fold tuning of FNN membership-function count and learning rate.
    kf = KFold(n_splits=4, shuffle=True, random_state=SEED)
    candidates = [(m, lr) for m in (2, 3) for lr in (5e-4, 1e-3, 2e-3)]
    rows = []
    for mfs, lr in candidates:
        fold_rmse = []
        for tr, va in kf.split(X_dev):
            model, sc, pred = train_one(X_dev[tr], y_dev[tr], X_dev[va], y_dev[va], mfs, lr)
            fold_rmse.append(rmse(y_dev[va], pred))
        rows.append({"mfs_per_feature": mfs, "learning_rate": lr,
                     "mean_cv_RMSE": float(np.mean(fold_rmse)),
                     "std_cv_RMSE": float(np.std(fold_rmse))})
        print(rows[-1])

    tuning = pd.DataFrame(rows).sort_values("mean_cv_RMSE")
    tuning.to_csv(os.path.join(OUT_DIR, "fnn_4fold_tuning.csv"), index=False)

    best = tuning.iloc[0]
    best_mfs = int(best["mfs_per_feature"])
    best_lr = float(best["learning_rate"])
    print("Best FNN:", best_mfs, best_lr)

    # Final FNN: internal validation is used only for early stopping; test remains untouched.
    Xtr, Xval, ytr, yval = train_test_split(X_dev, y_dev, test_size=0.15, random_state=SEED)
    model, sc, _ = train_one(Xtr, ytr, Xval, yval, best_mfs, best_lr)
    Xtest_s = preprocess_transform(X_test, sc)
    test_metrics, test_pred = evaluate(model, Xtest_s, y_test)

    high = y_test > 50
    high_metrics = {"n": int(high.sum()), "threshold": 50.0}
    if high.sum() >= 2:
        high_metrics.update({"high_" + k: v for k, v in metrics(y_test[high], test_pred[high]).items()})

    out = {"best_mfs_per_feature": best_mfs, "best_learning_rate": best_lr,
           "test_metrics": test_metrics, "high_collapse_gt_50": high_metrics,
           "split": "70% development / 30% independent test",
           "preprocessing": "log1p(Suction) then StandardScaler fitted only on training folds"}

    # Helper to convert numpy floats to Python floats for JSON serialization
    def convert_np_floats(obj):
        if isinstance(obj, np.float32):
            return float(obj)
        if isinstance(obj, dict):
            return {k: convert_np_floats(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [convert_np_floats(elem) for elem in obj]
        return obj

    # Apply conversion before dumping to JSON
    out_converted = convert_np_floats(out)

    with open(os.path.join(OUT_DIR, "fnn_results.json"), "w", encoding="utf-8") as f:
        json.dump(out_converted, f, indent=2)
    print("FNN test:", test_metrics)
    print("FNN CP > 50%:", high_metrics)

Development=420, independent test=180
Epoch 0001 | train MSE=199.8431 | val MSE=224.2701 | lr=5.00e-04
Epoch 0010 | train MSE=194.4488 | val MSE=222.6767 | lr=5.00e-04
Epoch 0020 | train MSE=190.9878 | val MSE=220.9453 | lr=4.98e-04
Epoch 0030 | train MSE=188.3248 | val MSE=219.2725 | lr=4.91e-04
Epoch 0040 | train MSE=190.4127 | val MSE=217.6410 | lr=4.81e-04
Epoch 0050 | train MSE=188.2348 | val MSE=216.0880 | lr=4.67e-04
Epoch 0060 | train MSE=187.6753 | val MSE=214.6201 | lr=4.48e-04
Epoch 0070 | train MSE=186.2140 | val MSE=213.2453 | lr=4.27e-04
Epoch 0080 | train MSE=184.8315 | val MSE=211.9381 | lr=4.02e-04
Epoch 0090 | train MSE=182.5368 | val MSE=210.7339 | lr=3.75e-04
Epoch 0100 | train MSE=182.0145 | val MSE=209.6319 | lr=3.46e-04
Epoch 0110 | train MSE=180.9950 | val MSE=208.6079 | lr=3.15e-04
Epoch 0120 | train MSE=184.7126 | val MSE=207.7025 | lr=2.83e-04
Epoch 0130 | train MSE=180.2254 | val MSE=206.8861 | lr=2.50e-04
Epoch 0140 | train MSE=177.3102 | val MSE=206.1808 |